# Refusal vector ablation demo

**Purpose:** educational mechanistic interpretability for safety research —
measure how strongly refusal is encoded as a linear direction, and how
orthogonal ablation changes residual geometry / generations.

**Not** an exploit guide. Use approved eval prompts under institutional policy.

Set `USE_MOCK = True` to run without downloading weights.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

USE_MOCK = True
MODEL_NAME = 'gpt2-small'  # e.g. 'Qwen/Qwen2.5-1.5B-Instruct' when ready
LAYER = 6
DEVICE = 'cpu'

In [ ]:
import numpy as np
from src.activation_harvest import harvest_mock, harvest_difference_means, DEFAULT_PAIRS
from src.vector_compute import difference_in_means, cosine, project_orthogonal

if USE_MOCK:
    data = harvest_mock(d_model=128, n=5, seed=0)
    model = None
else:
    from transformer_lens import HookedTransformer
    model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
    data = harvest_difference_means(model, DEFAULT_PAIRS, layer=LAYER, device=DEVICE)

r = difference_in_means(data['harmful_mean'], data['harmless_mean'])
print('layer', data['layer'], '||r||', np.linalg.norm(r))
print('cos(harmful, r)', cosine(data['harmful_mean'], r))
print('cos(harmless, r)', cosine(data['harmless_mean'], r))

In [ ]:
# Orthogonal ablation on the harmful mean (geometry check)
ablated = project_orthogonal(data['harmful_mean'], r)
print('cos after ablation (expect ~0):', cosine(ablated, r))

if model is not None:
    from src.weight_steering import make_orthogonal_ablation_hook, generate_with_hook
    hook_name, hook_fn = make_orthogonal_ablation_hook(r, LAYER)
    prompt = DEFAULT_PAIRS.harmful_prompts[0]
    print('BEFORE:', generate_with_hook(model, prompt)[:500])
    print('AFTER ABLATION:', generate_with_hook(model, prompt, hook_name, hook_fn)[:500])
else:
    print('Mock mode — skip generation. Set USE_MOCK=False after pip install -r requirements.txt')